In [1]:
import json
from pprint import pprint
import torch
# device = torch.device("cuda:1")
print("CUDA available:", torch.cuda.is_available())
print("Number of CUDA devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Number of CUDA devices: 2
Current CUDA device: 0
CUDA device name: NVIDIA RTX A5000


In [ ]:
# read csv file
import os
os.environ["OPENAI_API_KEY"] = ""
import pandas as pd
df = pd.read_csv('o4mini_annotation_verified(data2).csv')
df.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,t_result,t_corrected_version,t_category,t_notes,h_result,...,h_notes,final_result,final_corrected_version,final_notes,all_symptoms_extracted,missing_symptoms,llm_errors,num_groups,agreement,text_length
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",1,NaN,NaN,NaN,1,...,NaN,1,NaN,NaN,TRUE,NaN,NaN,4.0,True,110
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...","""No temporal information found""",No temporal information found,1,NaN,NaN,NaN,1,...,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,True,18
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",Symptoms with no original mention should be gr...,NaN,0,...,format,0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",LLM formatting issue; incorrect grouping; miss...,TRUE,NaN,NaN,2.0,False,13
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",1,NaN,NaN,NaN,0,...,Wrong order.,1,NaN,NaN,TRUE,NaN,NaN,2.0,False,260
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",1,NaN,NaN,NaN,1,...,NaN,1,NaN,NaN,TRUE,NaN,NaN,2.0,True,118


In [3]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 328.2+ KB


In [4]:
# check if "no temporal information" in final_corrected_version or llm_result_formatted if final_corrected_version is NaN and store in new column "no_temporal_info"
data['no_temporal_info'] = data.apply(lambda row:
    'no temporal information' in str(row['final_corrected_version']).lower() or
    ('final_corrected_version' in str(row['final_corrected_version']).lower() if
    pd.isna(row['final_corrected_version']) else False) or
    'no temporal information' in str(row['llm_result_formatted']).lower(),
    axis=1
)
data['no_temporal_info'].value_counts()
# get rows where no_temporal_info is False as new df
data_with_temporal_info = data[data['no_temporal_info'] == False]
data_with_temporal_info.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3794 entries, 0 to 5998
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3794 non-null   int64 
 1   symptom_text             3794 non-null   object
 2   symptom_list             3794 non-null   object
 3   llm_result               3794 non-null   object
 4   llm_result_formatted     3794 non-null   object
 5   final_corrected_version  654 non-null    object
 6   final_result             3794 non-null   int64 
 7   no_temporal_info         3794 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 240.8+ KB


/tmp/ipykernel_150198/4016686857.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['no_temporal_info'] = data.apply(lambda row:


In [5]:
# now lets take 10 random samples from data_with_temporal_info with random_state=2
sample = data_with_temporal_info.sample(n=5, random_state=2)
# sample.to_dict(orient='records')

In [6]:
generation_prompt = """
Ignore previous conversations.

Clinical Notes: {symptom_text}

Task:
    - Use the adverse effect list below:
      {symptom_list}
    - Reorder the list based on the sequence implied in the clinical notes.
    - If several adverse effects are mentioned **together without a clear temporal order**, group them into the same block (inside a single dictionary).
    - Do **not skip any terms** from the list — include all terms in the output.
    - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

Important Rules:
- Mention each symptom **only once** — at the first time it appears or becomes relevant.
- Do **NOT** treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
- If multiple symptoms are mentioned across multiple sentences **without clear timeline separation**, **group them together**.
- Only separate symptoms into different blocks when the clinical notes clearly describe a **sequence or timing** between them.

Finally, return your answer **only** in valid JSON format —
using a **list of dictionaries** to represent temporal progression and grouping, as shown below:
[
  {{ "Erythema": ["redness in neck"] }},
  {{
     "Pain in extremity": ["sore arm"],
     "Pruritus": ["itchy feeling"]
  }},
  {{ "Swelling": ["mild arm swelling"] }}
]
"""

In [7]:
verify_prompt = """

Given the original text and extracted symptoms below:

Original Text: {symptom_text}
Symptoms to Extract: {symptom_list}
Current Result: {initial_result}

Please evaluate the result based on these rules and assign 1 point for each satisfied rule (total possible score: 5):
1. The result is in valid JSON format with symptom groups ordered by time
2. All non-temporal conditions (marked as 'none') are in the last group
3. All symptoms from the symptom list are included - if not mentioned, they are in the last group as 'none'
4. Symptoms in the same group appear around the same time in the text
5. Groups follow temporal order from earliest to latest

Total Score: [score 0-5]

If score > 3:
    Return: {{'score': [score], 'output': {initial_result}}}

If score <= 3:
    Return: score, and feedback including specific suggestions for improvement and the initial result:
Output Format:
{{'score': int, 'output': json (if score > 3) or 'feedback': json (if score <= 3)}}
"""


In [ ]:


import os, json, re
from typing import Any, Optional, Dict, List, Tuple
from string import Template
import pandas as pd
from openai import OpenAI

client = OpenAI() 

# --------- Prompt templates ----------
generation_prompt_tpl = Template("""
Ignore previous conversations.

Clinical Notes:
$symptom_text

Task:
  - Use the adverse effect list below:
    $symptom_list
  - Reorder the list based on the sequence implied in the clinical notes.
  - If several adverse effects are mentioned together without a clear temporal order, group them into the same block (inside a single dictionary).
  - Do not skip any terms from the list — include all terms in the output.
  - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

Important Rules:
- Mention each symptom only once — at the first time it appears or becomes relevant.
- Do NOT treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", dates/times).
- If multiple symptoms are mentioned across multiple sentences without clear timeline separation, group them together.
- Only separate symptoms into different blocks when the notes clearly describe a sequence/timing between them.

OUTPUT (return ONLY a single JSON object; no extra text):
{[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]}
$feedback_block
""").substitute

verify_prompt_tpl = Template("""
Given the original text and extracted symptoms below:

Original Text:
$symptom_text

Symptoms to Extract:
$symptom_list

Current Result (JSON):
$initial_result

Scoring rubric (0–5), +1 each if:
1) The JSON is valid and groups are earliest→latest,
2) **Non-mentioned symptoms** should be presented as "none" in the **last** group,
3) Every symptom in the list appears exactly once overall,
4) Symptoms grouped together occur around the same time,
5) Group ordering follows the text's temporal cues.

Return ONLY one of the following JSON objects:
- If score >= 4: {"score": <int>, "feedback": ""}   # feedback can be empty string
- If score < 4:  {"score": <int>, "feedback": "<specific, actionable fixes>"}
""").substitute


# ---------- Helper: permissive JSON parser ----------
def parse_json_best_effort(text: str) -> Optional[Any]:
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (text.find("{"), text.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(text) - 1, start, -1):
            if text[end] in "}]":
                chunk = text[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None


# ---------- LLM call (Responses API)  ----------
def call_llm(messages: List[Dict[str, str]],
             model: str = "gpt-4.1",
             text: Optional[Dict] = None,            # e.g., {"format": {"type": "json_object"}}
             temperature: float = 0,
             max_output_tokens: int = 512,
             print_debug: bool = True) -> Tuple[str, Any]:
    kwargs = dict(
        model=model,
        input=messages,
        temperature=temperature,
        max_output_tokens=max_output_tokens
    )
    if text is not None:
        kwargs["text"] = text  # typed config for text output

    resp = client.responses.create(**kwargs)
    out_text = resp.output_text

    if print_debug:
        usage = getattr(resp, "usage", None)
        print("\n================ LLM CALL =================")
        print(f"Model: {model}")
        print("---- RAW OUTPUT ----")
        print(resp)
        print("---- USAGE ----")
        try:
            print(usage.model_dump() if usage else "(no usage)")
        except Exception:
            print(usage)
        print("===========================================\n")

    return out_text, resp


# ---------- Agent (ITERATIVE, prints every step) ----------
class TemporalExtractionAgent:
    def __init__(self, model_generate: str = "gpt-4.1", model_verify: Optional[str] = None, max_iterations: int = 3):
        self.model_generate = model_generate
        self.model_verify = model_verify or model_generate
        self.max_iterations = max_iterations

    def generate_once(self, symptom_text: str, symptom_list: List[str], prior_feedback: Optional[str]) -> Tuple[str, Optional[Any]]:
        symptom_list_json = json.dumps(symptom_list, ensure_ascii=False)
        feedback_block = "" if not prior_feedback else f"\nPrevious attempt feedback:\n{prior_feedback}\n"
        user_prompt = generation_prompt_tpl(
            symptom_text=symptom_text,
            symptom_list=symptom_list_json,
            feedback_block=feedback_block
        )
        messages = [
            {"role": "system", "content": "You are a medical temporal sequence analysis assistant. Respond ONLY with a single JSON object in the specified shape."},
            {"role": "user", "content": user_prompt},
        ]

        raw, _resp = call_llm(
            messages,
            model=self.model_generate,
            text={"format": {"type": "json_object"}},   # <-- FIX: valid type
            temperature=0,
            print_debug=True
        )

        parsed = parse_json_best_effort(raw)
        print("---- PARSED GENERATION ----")
        print(parsed if parsed is not None else "(parse failed)")
        print(raw)
        print("--------------------------------\n")
        return raw, parsed

    def verify_once(self, symptom_text: str, symptom_list: List[str], current_result_raw_or_parsed: Any) -> Tuple[str, Optional[Any]]:
        symptom_list_json = json.dumps(symptom_list, ensure_ascii=False)
        if isinstance(current_result_raw_or_parsed, (dict, list)):
            current_result_str = json.dumps(current_result_raw_or_parsed, ensure_ascii=False)
        else:
            current_result_str = str(current_result_raw_or_parsed)

        user_prompt = verify_prompt_tpl(
            symptom_text=symptom_text,
            symptom_list=symptom_list_json,
            initial_result=current_result_str
        )
        messages = [
            {"role": "system", "content": "You are a medical temporal sequence verification assistant. Respond ONLY with a single JSON object: {'score':<int>, 'feedback':<str>} (feedback required if score < 4)."},
            {"role": "user", "content": user_prompt},
        ]

        raw, _resp = call_llm(
            messages,
            model=self.model_verify,
            text={"format": {"type": "json_object"}},   # <-- FIX: valid type
            temperature=0,
            print_debug=True
        )

        parsed = parse_json_best_effort(raw)
        print("---- PARSED VERIFICATION ----")
        print(parsed if parsed is not None else "(parse failed)")
        print(raw)
        print("--------------------------------\n")
        return raw, parsed

    def process_sample(self, row, iterative: bool = True) -> Dict[str, Any]:
        """
        Returns EVERYTHING from all rounds, printing each step.
        {
          "attempts": [
             { "raw_gen": "...", "parsed_gen": <obj|None>, "raw_verify": "...", "parsed_verify": <obj|None> },
             ...
          ]
        }
        """
        symptom_text = row["symptom_text"]
        symptom_list = row["symptom_list"]

        attempts = []
        rounds = self.max_iterations if iterative else 1
        feedback = None
        last_gen_raw_or_parsed: Any = None

        for i in range(rounds):
            print(f"\n########## ITERATION {i+1}/{rounds} ##########")

            # generation
            gen_raw, gen_parsed = self.generate_once(symptom_text, symptom_list, prior_feedback=feedback)
            last_gen_raw_or_parsed = gen_parsed if gen_parsed is not None else gen_raw

            # verification
            ver_raw, ver_parsed = self.verify_once(symptom_text, symptom_list, last_gen_raw_or_parsed)

            attempts.append({
                "raw_gen": gen_raw,
                "parsed_gen": gen_parsed,
                "raw_verify": ver_raw,
                "parsed_verify": ver_parsed
            })

            # Early stop if score >= 4
            if isinstance(ver_parsed, dict) and isinstance(ver_parsed.get("score"), int) and ver_parsed["score"] >= 4:
                print(f"Verifier score >= 4 ({ver_parsed['score']}). Stopping early.\n")
                break

            # Feedback to next round (if provided)
            if isinstance(ver_parsed, dict) and "feedback" in ver_parsed:
                feedback = ver_parsed["feedback"]
                print("---- FEEDBACK FOR NEXT ROUND ----")
                print(feedback)
                print("----------------------------------\n")
            else:
                feedback = None

        return {"attempts": attempts}



In [9]:
# --------- Example batch driver ---------

agent = TemporalExtractionAgent(model_generate="gpt-4.1", max_iterations=3)

all_rows = []
for _, row in sample.iterrows():
    out = agent.process_sample(row, iterative=True)  # still iterative
    all_rows.append({
        "vaers_id": row["vaers_id"],
        "attempts_json": json.dumps(out["attempts"], ensure_ascii=False)  # everything the models said
    })
results_df = pd.DataFrame(all_rows)
results_df


########## ITERATION 1/3 ##########

================ LLM CALL =================
Model: gpt-4.1
---- RAW OUTPUT ----
Response(id='resp_03bcfcb0ab811a650068e557e2231c819e94022fc1ee6389c3', created_at=1759860706.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_03bcfcb0ab811a650068e557e2ceb4819e81fdfb174223c617', content=[ResponseOutputText(annotations=[], text='{\n  "sequence": [\n    {\n      "Headache": ["severe occipital headache (reported as patient rarely got headaches)"]\n    },\n    {\n      "Somnolence": ["somnolence (reported as not able to fall asleep despite feeling somnolent)"],\n      "Insomnia": ["insomnia (reported as not able to fall asleep despite feeling somnolent)"],\n      "Nausea": ["mild nausea"],\n      "Chills": ["chills"],\n      "Vaccination site pain": ["injection site soreness (vaccination site pain)"]\n    }\n  ]\n}', type='output_text', logprobs=[])],

,vaers_id,attempts_json
0,1465920,"[{""raw_gen"": ""{\n \""sequence\"": [\n {\n ..."
1,1552814,"[{""raw_gen"": ""{\n \""adverse_effect_sequence\""..."
2,2198919,"[{""raw_gen"": ""{\n \""sequence\"": [\n {\n ..."
3,1894487,"[{""raw_gen"": ""{\n \""adverse_event_sequence\"":..."
4,1272247,"[{""raw_gen"": ""{\n \""adverse_effect_sequence\""..."


In [24]:
for each in sample.llm_result_formatted.to_list():
    print("-------------------------------")
    print(each)

-------------------------------
[
  {
    "Headache": [
      "severe occipital headache"
    ]
  },
  {
    "Somnolence": [
      "somnolence"
    ],
    "Insomnia": [
      "insomnia"
    ],
    "Nausea": [
      "mild nausea"
    ],
    "Chills": [
      "chills"
    ],
    "Vaccination site pain": [
      "injection site soreness"
    ]
  }
]
-------------------------------
[
  {
    "Vaccination site mass": [
      "Large rump at the site of injection"
    ],
    "Vaccination site erythema": [
      "small spot of red at the injection site"
    ]
  },
  {
    "Vaccination site pain": [
      "pain during the injection"
    ]
  },
  {
    "Erythema": [
      "left arm was red with a big patch"
    ],
    "Night sweats": [
      "night sweats"
    ],
    "Lymph node pain": [
      "tenderness which is sometimes felt, sometimes not felt, in the lymph nodes of the right arm"
    ],
    "Pulse abnormal": [
      "feel and see a pronounced aortic pulse under her rib cage"
    ],
    "In

In [ ]:
# results_df.to_csv("temporal_extraction_results.csv", index=False)